In [ ]:
import cv2
import numpy as np
import glob

import cv2
import numpy as np
import os

CHECKERBOARD = (9, 6)
square_size = 0.025  # 25mm squares

save_folder = "calibration_images"
os.makedirs(save_folder, exist_ok=True)

cap = cv2.VideoCapture(0)   # <-- change to 1 if using USB camera

print("Press SPACE to capture a checkerboard image.")
print("Press ESC to finish capturing images.")

count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        print("Camera error!")
        break

    cv2.imshow("Calibration Capture", frame)

    key = cv2.waitKey(1)

    if key == 27:  # ESC
        break

    if key == 32:  # SPACE
        filename = f"{save_folder}/calib_{count}.jpg"
        cv2.imwrite(filename, frame)
        print("Saved:", filename)
        count += 1

cap.release()
cv2.destroyAllWindows()

CHECKERBOARD = (9, 6)  # inner corners
square_size = 0.020    # 25mm

# Prepare 3D points in real world space
objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)
objp *= square_size

objpoints = []
imgpoints = []

images = glob.glob('calibration_images/*.jpg')

for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)

    if ret:
        objpoints.append(objp)
        imgpoints.append(corners)

        cv2.drawChessboardCorners(img, CHECKERBOARD, corners, ret)
        cv2.imshow('img', img)
        cv2.waitKey(50)

cv2.destroyAllWindows()

# Calibrate
ret, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
    objpoints, imgpoints, gray.shape[::-1], None, None
)

np.save("camera_matrix.npy", camera_matrix)
np.save("dist_coeffs.npy", dist_coeffs)

print("Calibration complete.")
print("camera_matrix =\n", camera_matrix)
print("dist_coeffs =\n", dist_coeffs)
